# 12 — Pipeline Audit

This notebook demonstrates how to run and interpret the `audit_pipeline.py` script, which validates every installed stellar atmosphere model for unit correctness, bolometric consistency, and raw-vs-cube fidelity.

**Script location:** `tests/audit_pipeline.py`

### What the audit checks (per model)

| Check | Description |
|---|---|
| `units_standardized` flag | Every raw `.txt` file must have `# units_standardized = True` |
| Wavelength / flux units | Must be Å and erg/cm²/s/Å |
| Bolometric ratio | Raw file integral vs blackbody integral over same wavelength range |
| Wien peak position | Observed peak vs Planck prediction (UV-opacity and cool-star exemptions) |
| Negative flux | Count of negative flux values in raw files and cube nodes |
| Raw vs cube | Max relative error between raw spectrum and its cube node (10% threshold, tight-tolerance matched-pair method) |
| Collision detection | Duplicate (Teff, logg, [M/H]) nodes in `lookup_table.csv` |
| `fluxcube_library/` | Presence of per-variant cubes for models with extra axes |

In [1]:
import subprocess
import sys
import os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

## 1. Running the Audit

Run from the SED_Tools root. The script writes `audit_report.txt` and several diagnostic plots to the working directory.

In [6]:
# Run the audit — this may take several minutes for large installs
result = subprocess.run(
    [sys.executable, '../tests/audit_pipeline.py'],
    capture_output=True, text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])

0  (node: Teff=450 K  log g=4.50  [M/H]=0.00)
  Note: RAW VS CUBE: 21/30 nodes exceed 10% (expected — mean cube, variants in fluxcube_library/)
  Cube neg pts: 0 across 5 sampled nodes
  Bol ratio (cube): median=1.0716  std=0.0296  min=1.0025  max=1.0813
  Status: PASS

──────────────────────────────────────────────────────────────────────
Model: my_combined_grid
  Files       : 5212 total, 30 sampled (30 with valid Teff headers)
  Lookup      : 5212 entries -> 4629 unique nodes  (max 4 files/node)
  *** 444 NODES HAVE MULTIPLE FILES (last-write-wins in cube)
       Teff=4000.0 logg=3.5 [M/H]=0.0 - 4 files: Kurucz2003all_000171.txt, Kurucz2003all_000596.txt, NextGen2_001402.txt, hres_003596.txt
       Teff=4000.0 logg=4.0 [M/H]=0.0 - 4 files: Kurucz2003all_000172.txt, Kurucz2003all_000597.txt, NextGen2_001413.txt, hres_003597.txt
       Teff=4000.0 logg=4.5 [M/H]=0.0 - 4 files: Kurucz2003all_000173.txt, Kurucz2003all_000598.txt, NextGen2_001423.txt, hres_003598.txt
  Units flag  : OK (

## 2. Reading the Audit Report

`audit_report.txt` contains per-model sections with all check results and a summary at the end.

In [7]:
report_path = Path('audit_report.txt')
if report_path.exists():
    text = report_path.read_text()
    # Print just the summary lines
    lines = text.splitlines()
    in_summary = False
    for line in lines:
        if 'SUMMARY' in line.upper() or in_summary:
            in_summary = True
            print(line)
else:
    print("audit_report.txt not found — run the audit cell above first.")

SUMMARY: 14 PASS / 0 FAIL / 14 total


## 3. Running the Audit Programmatically

You can import and call `audit_model()` directly to audit a single model directory.

In [8]:
import sys
sys.path.insert(0, '../tests')  # make audit_pipeline importable

from audit_pipeline import audit_model, format_report
from sed_tools.api import SED
from pathlib import Path

local_catalogs = SED.query(include_remote=False)

if local_catalogs:
    model_name = local_catalogs[0].name
    model_dir  = Path(str(SED._model_root)) / model_name

    print(f"Auditing: {model_name}")
    result = audit_model(model_dir)

    print(f"  Files sampled:     {result['n_sampled']} / {result['n_files']}")
    print(f"  Units OK:          {result['all_standardized']}")
    print(f"  Cube exists:       {result['cube_exists']}")
    print(f"  Library exists:    {result['library_exists']}")
    print(f"  Collision nodes:   {result['n_collision_nodes']}")
    if result['bol_ratios_raw']:
        import numpy as np
        print(f"  Bol ratio median:  {np.median(result['bol_ratios_raw']):.4f}")
    if result['raw_vs_cube_max_err'] is not None:
        print(f"  Raw vs cube max err: {result['raw_vs_cube_max_err']:.4f}")
    if result['errors']:
        print(f"  Errors: {result['errors']}")

Auditing: Husfeld
  Files sampled:     30 / 448
  Units OK:          True
  Cube exists:       True
  Library exists:    True
  Collision nodes:   64
  Raw vs cube max err: 0.8571


## 4. Viewing the Diagnostic Plots

The audit generates several plots. Display them inline here.

In [5]:
plot_files = [
    ('audit_summary.png',       'Bolometric ratio by model'),
    ('wl_coverage.png',         'Wavelength coverage by model'),
    ('sed_comparison_hot.png',  'Hot star SED comparison'),
    ('sed_comparison_cold.png', 'Cold star SED comparison'),
]

for fname, title in plot_files:
    if os.path.exists(fname):
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.imshow(mpimg.imread(fname))
        ax.axis('off')
        ax.set_title(title)
        plt.tight_layout()
        plt.show()
    else:
        print(f"{fname} not found — run the audit first.")

/tmp/ipykernel_12755/4129218886.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Understanding Common Failures

| Failure | Likely cause | Fix |
|---|---|---|
| `units_standardized = False` | Spectra not yet processed by `spectra_cleaner` | Run `sed-tools rebuild` |
| Bol ratio >> 1 | Source data flux scale error (e.g. SVO NextGen T≥5000 K was 100× too large) | `spectra_cleaner` post-conversion renorm handles this automatically |
| Wien FAIL on hot compact objects | UV/EUV opacity shifts apparent SED peak — not a unit error | These models are in `WIEN_UV_OPACITY_WL_THRESHOLD` exemption |
| Raw vs cube > 10% with no library | Cube was built from wrong file list or has axis mismatch | Rebuild cube; check `FluxCube.from_file` axis order |
| Raw vs cube > 10% with library present | Expected — mean cube deviates from individual variants by design | Informational only, not a FAIL |